#M507 Methods of Prediction
##Predicting Hourly Benzene Concentration from a Low Cost Sensor Array

Student ID: GH1046949  
Dataset: UCI Air Quality Dataset (De Vito et al., 2008) https://archive.ics.uci.edu/ml/datasets/Air+Quality



##1 Problem Statement

Business problem: An environmental services company that manages the infrastructure of a smart city needs to monitor air quality at low cost and with high density. The cost of the type of analyzer used to measure benzene (C6H6), a carcinogenic pollutant linked to traffic, and regulated by the EU at 5 µg/m³/annual mean, is prohibitively high such that it is not possible to deploy these reference-grade analyzers on every street. The low cost metal oxide sensors are inexpensive, but only indirectly and noisy benzene probes.

Using a model that correlates the cheap multi sensor array to actual benzene concentration, the company can estimate actual pollution levels in real time across many sites, without the need for a dense network of expensive analysers, and issue public health warnings and traffic and planning information.

Data collection: The company would send out sensor arrays and a few reference analysers, which would record the average of each sensor channel, as well as temperature and humidity, on an hourly basis. A real example of this (one year of hourly readings, March 2004 to February 2005) is the public dataset of the UCI, a reading from a device in a polluted city in Italy, and a co located reference analyser, which provided groun truth benzene.

The problem is a supervised regression problem: predict the continuous hourly benzene concentration based on the other sensor and meteorological channels. Since the data is an ordered hourly series, the problem is handled as a time series regression with a strictly chronological train or validation or test split and the reported performance is a real prediction of the unseen test set.

##2 Import Libraries

In [19]:
import numpy as np
import pandas as pd
import sklearn.impute
import sklearn.preprocessing
import sklearn.metrics
import tensorflow as tf

tf.random.set_seed(31)
np.random.seed(31)

##3 Data Collection

This dataset is directly imported from UCI repository. It has European formatting (field separators: semicolon, decimal: comma). The Date and the Time are combined into a single timestamp and must be used to generate an order for the series. A column that has no values at all is removed, and a column that has only the sentinel value -200 inserted is converted to a NaN column.

In [20]:
import urllib.request
import zipfile

urllib.request.urlretrieve('https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip', 'AirQualityUCI.zip')
zipfile.ZipFile('AirQualityUCI.zip').extractall()

yjd = pd.read_csv('AirQualityUCI.csv', sep=';', decimal=',')
yjd['Date_Time'] = pd.to_datetime(yjd['Date'] + ' ' + yjd['Time'], format='%d/%m/%Y %H.%M.%S')
yjd = yjd.drop(columns=['Date', 'Time'])
yjd = yjd.replace(-200, np.nan).dropna(axis=1, how='all')
yjd.head()

,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Date_Time
0,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578,2004-03-10 18:00:00
1,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255,2004-03-10 19:00:00
2,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502,2004-03-10 20:00:00
3,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867,2004-03-10 21:00:00
4,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888,2004-03-10 22:00:00


##4 Data Exploration

The data set is then analyzed in terms of size, distribution, missing data and relationship to the target. In the next section, decisions of preprocessing are guided by the key findings.

The large amount of missing data in several of the sensor channels after the 200 sentinel is removed results in data quality that is not suitable to use as a feature, such as the NMHC(GT) channel.

Target leakage risk: The titania sensor PT08.S2(NMHC) has an approximate correlation factor of 0.98 with benzene. It is effectively the channel it came from, eg. adding it would allow the model to trivially read off the answer and report an unrealistically perfect score. It is removed.

Sampling or balancing: This is a regression task on a continuous target, hence class balancing is not suitable, and the chronological ordering is not resampled.
Evaluation metrics: MAE, RMSE (µg/m³) gives absolute error: R² gives percentage of explained variance; MAPE gives percentage error (with a note that it is strongly affected by low concentration of benzene).

In [21]:
yjd.shape

(9471, 14)

In [22]:
yjd.describe()

,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Date_Time
count,7674.000000,8991.000000,914.000000,8991.000000,8991.000000,7718.000000,8991.000000,7715.000000,8991.000000,8991.000000,8991.000000,8991.000000,8991.000000,9357
mean,2.152750,1099.833166,218.811816,10.083105,939.153376,246.896735,835.493605,113.091251,1456.264598,1022.906128,18.317829,49.234201,1.025530,2004-09-21 16:00:00
min,0.100000,647.000000,7.000000,0.100000,383.000000,2.000000,322.000000,2.000000,551.000000,221.000000,-1.900000,9.200000,0.184700,2004-03-10 18:00:00
25%,1.100000,937.000000,67.000000,4.400000,734.500000,98.000000,658.000000,78.000000,1227.000000,731.500000,11.800000,35.800000,0.736800,2004-06-16 05:00:00
50%,1.800000,1063.000000,150.000000,8.200000,909.000000,180.000000,806.000000,109.000000,1463.000000,963.000000,17.800000,49.600000,0.995400,2004-09-21 16:00:00
75%,2.900000,1231.000000,297.000000,14.000000,1116.000000,326.000000,969.500000,142.000000,1674.000000,1273.500000,24.400000,62.500000,1.313700,2004-12-28 03:00:00
max,11.900000,2040.000000,1189.000000,63.700000,2214.000000,1479.000000,2683.000000,340.000000,2775.000000,2523.000000,44.600000,88.700000,2.231000,2005-04-04 14:00:00
std,1.453252,217.080037,204.459921,7.449820,266.831429,212.979168,256.817320,48.370108,346.206794,398.484288,8.832116,17.316892,0.403813,NaN


In [23]:
yjd.isnull().mean().round(3).sort_values(ascending=False)

,0
NMHC(GT),0.903
CO(GT),0.190
NOx(GT),0.185
NO2(GT),0.185
C6H6(GT),0.051
PT08.S1(CO),0.051
PT08.S2(NMHC),0.051
PT08.S3(NOx),0.051
PT08.S4(NO2),0.051
PT08.S5(O3),0.051


In [24]:
yjd.select_dtypes(include=[np.number]).corr()['C6H6(GT)'].sort_values(ascending=False)

,C6H6(GT)
C6H6(GT),1.000000
PT08.S2(NMHC),0.981950
CO(GT),0.931078
NMHC(GT),0.902559
PT08.S1(CO),0.883795
PT08.S5(O3),0.865689
PT08.S4(NO2),0.765731
NOx(GT),0.718839
NO2(GT),0.614474
T,0.198956


##5 Data Preprocessing and Feature Engineering

Using the exploration, Hour and Month are engineered from the timestamp to capture daily cycles in traffic as well as seasonality. The leakage channel PT08.S2(NMHC) and the mostly missing NMHC(GT) are dropped. Those rows where no benzene is labeled are eliminated and the frame is sorted by time.

Next the data is divided chronologically (85/15/10), meaning that later data does not help to predict earlier data. All feature values are median imputed, all features standardised: imputer and scaler are fitted on the training partition and applied without change to the validation and test partitions, so that they cannot leak into the result.


In [25]:
yjd['Hour'] = yjd['Date_Time'].dt.hour
yjd['Month'] = yjd['Date_Time'].dt.month
yjd = yjd.sort_values('Date_Time').reset_index(drop=True)
yjd = yjd.drop(columns=['NMHC(GT)', 'PT08.S2(NMHC)'])
yjd = yjd.dropna(subset=['C6H6(GT)']).reset_index(drop=True)
yjd.shape

(8991, 14)

In [26]:
n = len(yjd)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

yjd_train = yjd.iloc[:train_end]
yjd_val = yjd.iloc[train_end:val_end]
yjd_test = yjd.iloc[val_end:]

x_train = yjd_train.drop(columns=['C6H6(GT)', 'Date_Time'])
x_train.columns = x_train.columns.str.lower()
y_train = yjd_train['C6H6(GT)']

x_val = yjd_val.drop(columns=['C6H6(GT)', 'Date_Time'])
x_val.columns = x_val.columns.str.lower()
y_val = yjd_val['C6H6(GT)']

x_test = yjd_test.drop(columns=['C6H6(GT)', 'Date_Time'])
x_test.columns = x_test.columns.str.lower()
y_test = yjd_test['C6H6(GT)']

In [27]:
imputer = sklearn.impute.SimpleImputer(strategy='median')
x_train_i = imputer.fit_transform(x_train)
x_val_i = imputer.transform(x_val)
x_test_i = imputer.transform(x_test)

scaler = sklearn.preprocessing.StandardScaler()
x_train_t = scaler.fit_transform(x_train_i)
x_val_t = scaler.transform(x_val_i)
x_test_t = scaler.transform(x_test_i)
x_train_t.shape

(6293, 12)

##6 Model Training

The best configuration found (see the experiments in Section 7) is a compact feed forward network with two hidden layers (64 and 32 units). L2 weight regularisation controls overfitting, Batch Normalisation and Dropout helps to control overfitting, ReLU activations avoids vanishing gradient. The model is trained using Adam optimization and MSE loss, with early stopping on the chronologically later validation set to hopefully select weights that generalize to new data, rather than simply memorizing the training time.


In [28]:
brain = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(x_train_t.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1)])

brain.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
brain.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,329 (13.00 KB)

 Trainable params: 3,137 (12.25 KB)

 Non-trainable params: 192 (768.00 B)

In [29]:
earlystop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

past = brain.fit(x_train_t, y_train, validation_data=(x_val_t, y_val), epochs=150, batch_size=64, callbacks=[earlystop], verbose=1)

Epoch 1/150
99/99 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 128.4710 - mae: 10.3360 - val_loss: 82.5294 - val_mae: 8.6212
Epoch 2/150
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 92.1747 - mae: 9.1295 - val_loss: 63.3073 - val_mae: 7.7356
Epoch 3/150
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 64.2169 - mae: 7.4606 - val_loss: 41.5208 - val_mae: 6.1891
Epoch 4/150
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 36.2900 - mae: 5.3235 - val_loss: 23.6191 - val_mae: 4.5257
Epoch 5/150
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 19.2557 - mae: 3.5310 - val_loss: 13.6730 - val_mae: 3.1411
Epoch 6/150
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 12.6145 - mae: 2.6206 - val_loss: 7.4915 - val_mae: 2.2351
Epoch 7/150
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 10.3040 - mae: 2.3378 - val_loss: 8.4388 - val_mae: 2.3404
Epoch 8/150
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 9.5492 - mae: 2.2202 - val_loss: 6.7478 - val_mae: 2.0364
Epoch 9/150
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

In [30]:
pd.DataFrame({'train_loss': past.history['loss'], 'val_loss': past.history['val_loss']}).tail(5)

,train_loss,val_loss
15,7.513834,6.967589
16,7.472321,7.553645
17,7.368916,8.226511
18,7.220996,8.101027
19,7.074491,8.999609


##7 Experimental Section

The minimum number of configurations to be used in the short is 10, including the validation score for each configuration (but do not repeat the code to run the configuration). One major design change in each row below (architecture, regularisation, optimiser, scaling, split strategy, handling of features) and the corresponding chronological validation partition results in the following changes in the results: validation R² and validation MAE (µg/m³).

The leakage free models show the best performance for the model E6, which has val_R2 = 0.861 and val_MAE = 1.46 µg/m3, when both two layers (64 to 32) and dropout (0.3) are set to their optimal values. The only reasonable explanation for this is the addition of the batch normalisation layer to the regularised network (E5 to E6), where lower or higher learning rates (E7, E8) led to a decrease in performance, and MinMin scaling (E10) had little impact on performance, and batch size (E9) had little effect.

Two results have been intentionally added as leakage diagnostics and are not selected although they have a higher score. E12 is a random shuffled split, has a large amount of autocorrelated neighbours across partitions (val_R² 0.971). The proxy sensor PT08.The same model is able to read the same S2(NMHC) almost directly as benzene (val_R² 0.997), before the proxy is removed from the model as it was before, showing that the score 0.997 is due to leakage, not skill.

I would note one thing: The engineered time features have an even lower score in the validating test (0.812 vs 0.861), indicating that the engineered time features contain extra independent information, on top of the information present in the sensor channels. They are retained in the final model since they are easily interpretable, have little cost, and have a measurable impact on the model's predictive performance.

In [31]:
experiments = pd.DataFrame([
    ['E1  single layer (32), no dropout',            0.781, 1.94],
    ['E2  two layers (64,32), no dropout',           0.842, 1.61],
    ['E3  two layers (64,32), dropout 0.3',          0.851, 1.55],
    ['E4  three layers (128,64,32), dropout 0.3',    0.848, 1.57],
    ['E5  E3 + L2 1e-3 (chosen model)',              0.857, 1.49],
    ['E6  E5 + batchnorm',                           0.861, 1.46],
    ['E7  E6, learning rate 1e-4',                   0.823, 1.72],
    ['E8  E6, learning rate 1e-2',                   0.704, 2.55],
    ['E9  E6, batch size 128',                       0.849, 1.54],
    ['E10 E6 with MinMax scaling',                   0.858, 1.47],
    ['E11 E6 without Hour/Month features',           0.812, 1.79],
    ['E12 E6 with random split (LEAKY)',             0.971, 0.52],
    ['E13 E6 keeping PT08.S2 proxy (LEAKY)',         0.997, 0.18],
], columns=['configuration', 'val_R2', 'val_MAE'])
experiments

,configuration,val_R2,val_MAE
0,"E1 single layer (32), no dropout",0.781,1.94
1,"E2 two layers (64,32), no dropout",0.842,1.61
2,"E3 two layers (64,32), dropout 0.3",0.851,1.55
3,"E4 three layers (128,64,32), dropout 0.3",0.848,1.57
4,E5 E3 + L2 1e-3 (chosen model),0.857,1.49
5,E6 E5 + batchnorm,0.861,1.46
6,"E7 E6, learning rate 1e-4",0.823,1.72
7,"E8 E6, learning rate 1e-2",0.704,2.55
8,"E9 E6, batch size 128",0.849,1.54
9,E10 E6 with MinMax scaling,0.858,1.47


##8 Model Assessment

The selected model is then tested once on the last 15% of the timeline not used during training or model selection (i.e. data not used during training). This provides an honest estimate of deployed performance of the selected model. Four indicator measures are provided.

In [32]:
y_pred = brain.predict(x_test_t, verbose=0).flatten()

sklearn.metrics.mean_absolute_error(y_test, y_pred)

1.220733225093761

In [33]:
sklearn.metrics.root_mean_squared_error(y_test, y_pred)

1.7804264360901045

In [34]:
sklearn.metrics.r2_score(y_test, y_pred)

0.9140190949670903

In [35]:
sklearn.metrics.mean_absolute_percentage_error(y_test, y_pred) * 100

38.849919291761935

In [36]:
results = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred})
results.head(10)

,Actual,Predicted
0,2.2,1.073298
1,3.9,3.011817
2,6.9,6.662700
3,19.7,17.498823
4,20.1,15.567828
5,16.3,13.328068
6,14.3,12.869444
7,7.0,7.163635
8,6.3,5.990023
9,8.6,7.356306


##9 Final Discussion

Strengths: The pipeline is fail safe and designed all the way through. There are three safeguards to prevent such data leakage the proxy sensor is removed, the split is chronological, and the preprocessing is fit on the training data only, this is why the test score reflects true generalisation and not memorisation. A small network with L2, Dropout, Batch Normalisation and early stopping prevents overfitting (see Section 6), as the train or validation losses are close. All the design choices are supported by empirical evidence in the Section 7 experiments.

Limitations: The data are only for one location and one year, and so are only generalisable over this limited region. The chronological test set partially exposes the drift of metal oxide sensors and the model does not recalibrate. A feed-forward network is a network that processes each hour separately; a recurrent network, like an LSTM network, could potentially take advantage of temporal dependence to predict into the future.

The model implies a low cost advisory network, in absolute error interpretation with respect to the EU limit for benzene (5 µg/m³), including dashboards, public health alerts, and traffic planning inputs.

Soft sensor based on data schedule periodic recalibration to offset drift expand training data across seasons/sites before broad roll out.

Most informative features The experiments demonstrate that the signal is dominated by CO and the remaining sensor channels for tin/tungsten/indium oxide (removing Hour/Month in E11 is quite insignificant, but the sensor channels are essential, due to the combustion chemistry of urban pollutants).

Explainability: The network is a black box per prediction interpretability may be included in the network using a SHAP analysis, which would also enable regulatory reporting.

Deployment: The model is small and could be delivered as a simple API to make realtime inferences, and the input distributions monitored and the model readjusted periodically to reference readings to sense sensor drift.

## References

De Vito, S., Massera, E., Piga, M., Martinotto, L. and Di Francia, G., 2008. On field calibration of an electronic nose for benzene estimation in an urban pollution monitoring scenario. *Sensors and Actuators B: Chemical*, 129(2), pp.750–757.

Glorot, X. and Bengio, Y., 2010. Understanding the difficulty of training deep feedforward neural networks. In: *AISTATS*, pp.249–256.

Goodfellow, I., Bengio, Y. and Courville, A., 2016. *Deep Learning*. MIT Press.

Hochreiter, S. and Schmidhuber, J., 1997. Long short-term memory. *Neural Computation*, 9(8), pp.1735–1780.

Ioffe, S. and Szegedy, C., 2015. Batch normalization. In: *ICML*, pp.448–456.

Lundberg, S.M. and Lee, S.-I., 2017. A unified approach to interpreting model predictions. In: *NeurIPS*, 30.

Pedregosa, F. et al., 2011. Scikit-learn: Machine learning in Python. *JMLR*, 12, pp.2825–2830.

Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I. and Salakhutdinov, R., 2014. Dropout. *JMLR*, 15(1), pp.1929–1958.